<a href="https://colab.research.google.com/github/Syed-Irtaza/CrewLogix_GenAI_Assignment/blob/main/RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install faiss-cpu sentence-transformers pypdf python-docx openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 69.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.9/343.9 kB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 21.1 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from google.colab import files

uploaded = files.upload()
file_path = list(uploaded.keys())[0]

print("Uploaded file:", file_path)

Saving The_Plan_of_the_Giza_Pyramids.pdf to The_Plan_of_the_Giza_Pyramids (1).pdf
Uploaded file: The_Plan_of_the_Giza_Pyramids (1).pdf


In [ ]:
import os
from pypdf import PdfReader
from docx import Document

def extract_text(file_path):
    extension = os.path.splitext(file_path)[1].lower()

    if extension == ".pdf":
        reader = PdfReader(file_path)
        text = ""
        for page in reader.pages:
            text += page.extract_text() + "\n"
        return text

    elif extension == ".docx":
        doc = Document(file_path)
        text = ""
        for paragraph in doc.paragraphs:
            text += paragraph.text + "\n"
        return text

    elif extension == ".txt":
        with open(file_path, "r", encoding="utf-8") as file:
            return file.read()

    else:
        raise ValueError("Only PDF, DOCX, and TXT files are supported.")

document_text = extract_text(file_path)

print(document_text[:1000])

The Plan of the Giza Pyramids  1 
 
The Plan of the Giza Pyramids 
by John A.R. Legon 
Revision Date:   22-11-2019 
My initial findings on the Giza Site Plan of Three Pyramids were first briefly summarized in a 
pamphlet  published by the Archaeology Society of Staten Island in 1979.1  Subsequently, I 
described the most significant features of the integrated site plan in several articles in the journals 
Discussions in Egyptology2 and Göttinger Miszellen.3  Further research has shown that many 
more factors must be taken into account, without alterating the basic framework of the dimensions 
in royal cubits which I described in 1979. The following text is  based on my article in Discussions 
in Egyptology Vol. 10, but now includes much new material and new illustrations.  In addition, 
acount has been taken of the extensive survey work carried out since 2012 by the late Glen Dash, 
and the positions of the corners of the three pyramids have been placed in the coordinate system 
initia

In [ ]:
def create_chunks(text, chunk_size=500, overlap=100):
    words = text.split()
    chunks = []

    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk = words[start:end]
        chunks.append(" ".join(chunk))

        start = end - overlap

    return chunks

chunks = create_chunks(document_text)

print("Total chunks:", len(chunks))
print(chunks[0])

Total chunks: 16
The Plan of the Giza Pyramids 1 The Plan of the Giza Pyramids by John A.R. Legon Revision Date: 22-11-2019 My initial findings on the Giza Site Plan of Three Pyramids were first briefly summarized in a pamphlet published by the Archaeology Society of Staten Island in 1979.1 Subsequently, I described the most significant features of the integrated site plan in several articles in the journals Discussions in Egyptology2 and Göttinger Miszellen.3 Further research has shown that many more factors must be taken into account, without alterating the basic framework of the dimensions in royal cubits which I described in 1979. The following text is based on my article in Discussions in Egyptology Vol. 10, but now includes much new material and new illustrations. In addition, acount has been taken of the extensive survey work carried out since 2012 by the late Glen Dash, and the positions of the corners of the three pyramids have been placed in the coordinate system initiated by

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

chunk_embeddings = embedding_model.encode(chunks)

chunk_embeddings = np.array(chunk_embeddings).astype("float32")

print("Embedding shape:", chunk_embeddings.shape)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding shape: (16, 384)


In [ ]:
import faiss

dimension = chunk_embeddings.shape[1]

# Normalize embeddings for cosine similarity
faiss.normalize_L2(chunk_embeddings)

# IndexFlatIP uses inner product.
# With normalized vectors, inner product behaves like cosine similarity.
index = faiss.IndexFlatIP(dimension)

index.add(chunk_embeddings)

print("Total vectors stored in FAISS:", index.ntotal)

Total vectors stored in FAISS: 16


In [ ]:
def search_similar_chunks(query, top_k=3):
    query_embedding = embedding_model.encode([query])
    query_embedding = np.array(query_embedding).astype("float32")

    faiss.normalize_L2(query_embedding)

    scores, indexes = index.search(query_embedding, top_k)

    retrieved_chunks = []

    for i, chunk_index in enumerate(indexes[0]):
        retrieved_chunks.append({
            "chunk": chunks[chunk_index],
            "score": scores[0][i]
        })

    return retrieved_chunks

In [ ]:
import os
from google.colab import userdata
from openai import OpenAI

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

client = OpenAI()

In [ ]:
def ask_rag(query):
    retrieved = search_similar_chunks(query, top_k=3)

    context = ""

    for item in retrieved:
        context += item["chunk"] + "\n\n"

    prompt = f"""
You are a helpful assistant. Answer the question using only the given context.

Context:
{context}

Question:
{query}

Answer:
"""

    response = client.responses.create(
        model="gpt-4o-mini",
        input=prompt
    )

    return response.output_text

In [ ]:
query = input("Ask your question: ")

answer = ask_rag(query)

print("\nFinal Answer:\n")
print(answer)

Ask your question: what this document is about?

Final Answer:

This document discusses the architectural and dimensional planning of the Giza Pyramids, particularly focusing on the layout and design principles underlying the arrangement of the three pyramids. It critiques the conventional view of their function as tombs, positing instead that they may have served as a center for ancient Egyptian initiation rites. The analysis includes historical references, survey data, and geometric relationships among the pyramids, suggesting a meticulous design with significant mathematical properties.
